# Pipeline - Extract + Transform

```mermaid
flowchart TB
    subgraph EXTRACT["EXTRACT — parcours propre à chaque source"]
        direction TB

        subgraph NEWSDATA["NewsData.io — API REST JSON"]
            direction LR
            N0[("API /api/1/latest")]
            N1["Requests<br/>clé API, filtres, pagination, retries"]
            N2["Contrôle<br/>article_id + texte + image_url"]
            N3["download_image()<br/>image + SHA-256 + preuve URL"]
            N4[("Lot brut newsdata<br/>JSON + images")]
            N0 --> N1 --> N2 --> N3 -->|"save_json_records()"| N4
        end

        subgraph POLITIFACT["PolitiFact — flux RSS"]
            direction LR
            P0[("RSS fact-checks")]
            P1["Requests + Feedparser<br/>lecture XML"]
            P2["Sélection<br/>id + summary/content + thumbnail"]
            P3["download_image()<br/>image + SHA-256 + preuve URL"]
            P4[("Lot brut politifact<br/>JSON + images")]
            P0 --> P1 --> P2 --> P3 -->|"save_json_records()"| P4
        end

        subgraph FAKEDDIT["Fakeddit — dataset TSV"]
            direction LR
            F0[("multimodal_train.tsv")]
            F1["csv.DictReader<br/>lecture ligne par ligne"]
            F2["Filtre<br/>id + clean_title + hasImage + image_url"]
            F3["download_image()<br/>image + SHA-256 + preuve URL"]
            F4[("Lot brut fakeddit<br/>JSON + images")]
            F0 --> F1 --> F2 --> F3 -->|"save_json_records()"| F4
        end

        subgraph CONVERSATION["The Conversation France — Atom + HTML"]
            direction LR
            T0[("Flux articles.atom")]
            T1["Requests + Feedparser<br/>découverte des URLs"]
            T2["Requests<br/>téléchargement page HTML"]
            T3["Beautiful Soup<br/>h1, articleBody, time, author, og:image"]
            T4["download_image()<br/>image + SHA-256 + preuve URL"]
            T5[("Lot brut theconversation<br/>JSON + images")]
            T0 --> T1 --> T2 --> T3 --> T4 -->|"save_json_records()"| T5
        end
    end

    subgraph MAPPING["TRANSFORM — mapping propre à chaque source"]
        direction TB
        MN["transform_newsdata()<br/>repli content → description → ai_summary<br/>langue french → fr"]
        MP["transform_politifact()<br/>nettoyage HTML + conservation du verdict"]
        MF["transform_fakeddit()<br/>date Unix + conservation labels 2/3/6"]
        MT["transform_theconversation()<br/>champs HTML déjà extraits + langue fr"]
    end

    N4 --> MN
    P4 --> MP
    F4 --> MF
    T5 --> MT

    MN --> C0
    MP --> C0
    MF --> C0
    MT --> C0

    subgraph COMMON["TRANSFORM — traitement commun après convergence"]
        direction TB
        C0["Contrat provisoire commun<br/>18 champs"]
        C1["validate_record()<br/>identifiant, texte, URLs, dates, langue, labels"]
        C2["validate_image()<br/>chemin, nom, signature, taille, SHA-256, provenance"]
        C3["deduplicate_candidates()<br/>publication_id OU source_url + image_sha256"]
        C4["make_dataframe()<br/>typage, tri stable, ordre des colonnes"]
        C0 --> C1 -->|"valide"| C2 -->|"image valide"| C3 -->|"occurrence retenue"| C4
    end

    C1 -->|"invalide"| R[("invalid_records.jsonl")]
    C2 -->|"image invalide"| R
    C3 -->|"doublon écarté"| R

    C4 --> O1[("publications.parquet<br/>contrat Load — 18 colonnes")]
    C4 --> O2[("publications.jsonl")]
    C4 --> O3[("transformation_manifest.json<br/>paramètres + hashes + métriques")]
    C4 --> O4[("logs/transform.log")]

    classDef source fill:#fff3dd,stroke:#ad6500,color:#332100
    classDef process fill:#eaf2ff,stroke:#2056a8,color:#102a4c
    classDef data fill:#eaf8ef,stroke:#18794e,color:#0f3b29
    classDef reject fill:#fdecec,stroke:#b42318,color:#5c1510

    class N0,P0,F0,T0 source
    class N1,N2,N3,P1,P2,P3,F1,F2,F3,T1,T2,T3,T4,MN,MP,MF,MT,C0,C1,C2,C3,C4 process
    class N4,P4,F4,T5,O1,O2,O3,O4 data
    class R reject
```

# Schéma conceptuel et contrat de données pour le chargement

## Objectif

Ce document relie deux niveaux complémentaires :

1. le **contrat d’entrée de la phase Load**, qui est le fichier Parquet plat produit par Transform ;
2. le **modèle cible logique**, qui répartit ce contrat dans cinq tables reliées par des clés primaires et étrangères.

Le grain du contrat est : **une ligne par publication unique, associée à exactement une image locale validée**. Les labels restent facultatifs. Le modèle ne constitue pas encore un script SQL propre à Snowflake : il fixe les données, les clés et les relations que le futur chargement devra respecter.

## Contrat d’entrée de la phase Load

La phase Load reçoit `data/processed/publications.parquet`. Ce fichier contient les 18 colonnes documentées plus bas et possède une clé métier :

- **PK logique :** `publication_id` ;
- **grain :** une publication multimodale dédupliquée ;
- **image obligatoire :** les cinq champs `image_*` sont renseignés sur chaque ligne ;
- **label facultatif :** les trois champs de label sont soit tous renseignés, soit tous absents ;
- **clé secondaire de dédoublonnage :** le couple `source_url + image_sha256`.

Le fichier Parquet reste volontairement aplati pour faciliter son contrôle et son chargement. La phase Load pourra le répartir dans le modèle cible ci-dessous sans perdre de colonne.

## Modèle cible logique de la phase Load

```mermaid
erDiagram
    SOURCE_ACQUISITION ||--o{ LOT_EXTRACTION : "produit"
    LOT_EXTRACTION ||--o{ PUBLICATION : "contient"
    PUBLICATION ||--|| IMAGE : "possede"
    PUBLICATION ||--o| LABEL_SOURCE : "peut recevoir"

    SOURCE_ACQUISITION {
        string source_id PK "NOT NULL - SHA-256 stable de source_name"
        string source_name UK "NOT NULL - nom acquisition"
    }

    LOT_EXTRACTION {
        string lot_id PK "NOT NULL - SHA-256 source et collecte"
        string source_id FK "NOT NULL - vers SOURCE_ACQUISITION"
        datetime collected_at "NOT NULL - UTC"
    }

    PUBLICATION {
        string publication_id PK "NOT NULL - identifiant stable"
        string lot_id FK "NOT NULL - vers LOT_EXTRACTION"
        string source_domain "NOT NULL - domaine editorial ou cible"
        string source_url "NOT NULL - URL HTTP ou HTTPS"
        string title "NULL autorise si text present"
        text text "NULL autorise si title present"
        datetime published_at "NOT NULL - UTC"
        string language "NOT NULL - ISO 639-1"
        string author "NULL autorise"
    }

    IMAGE {
        string publication_id PK, FK "NOT NULL - vers PUBLICATION"
        string image_url "NOT NULL - URL distante"
        string image_path "NOT NULL - chemin valide"
        integer image_size_bytes "NOT NULL - valeur superieure a zero"
        string image_sha256 "NOT NULL - 64 caracteres hexadecimaux"
        string image_provenance_status "NOT NULL - valeur controlee"
    }

    LABEL_SOURCE {
        string publication_id PK, FK "NOT NULL - vers PUBLICATION"
        string source_label_raw "NOT NULL si ligne presente"
        string source_label_scheme "NOT NULL si ligne presente"
        string label_provenance "NOT NULL si ligne presente"
    }
```

### Lecture des relations et des clés

- `SOURCE_ACQUISITION.source_id` est la **PK** de la source. `source_name` possède aussi une contrainte d’unicité métier.
- `LOT_EXTRACTION.lot_id` est la **PK** du lot et `source_id` est une **FK** vers sa source d’acquisition.
- `PUBLICATION.publication_id` est la **PK** de la publication et `lot_id` est une **FK** vers le lot qui a fourni l’occurrence conservée après dédoublonnage.
- `IMAGE.publication_id` est à la fois sa **PK** et une **FK** vers `PUBLICATION`. Cette clé partagée impose une relation 1–1 : une publication chargée possède exactement une image.
- `LABEL_SOURCE.publication_id` est également une **PK/FK**. Une ligne n’est créée que si les trois informations du label existent, d’où la cardinalité 0–1.
- Une source peut produire plusieurs lots ; un lot peut contenir plusieurs publications.

### Identifiants calculés pendant Load

Deux clés techniques ne sont pas ajoutées au Parquet, car elles sont dérivables de manière reproductible :

```text
source_id = SHA-256(normaliser(source_name))
lot_id    = SHA-256(source_id + "|" + collected_at_UTC)
```

La même entrée produit donc toujours la même clé. `publication_id`, déjà produit par Transform, n’est pas recalculé pendant Load.

### Mapping du Parquet vers les tables cibles

| Table cible | PK | FK | Colonnes provenant du Parquet |
|---|---|---|---|
| `SOURCE_ACQUISITION` | `source_id` calculé | — | `source_name` |
| `LOT_EXTRACTION` | `lot_id` calculé | `source_id` | `collected_at` |
| `PUBLICATION` | `publication_id` | `lot_id` | `source_domain`, `source_url`, `title`, `text`, `published_at`, `language`, `author` |
| `IMAGE` | `publication_id` | `publication_id → PUBLICATION` | `image_url`, `image_path`, `image_size_bytes`, `image_sha256`, `image_provenance_status` |
| `LABEL_SOURCE` | `publication_id` | `publication_id → PUBLICATION` | `source_label_raw`, `source_label_scheme`, `label_provenance` |

Les 18 colonnes du contrat sont ainsi toutes utilisées. `source_id` et `lot_id` sont des colonnes techniques supplémentaires ; `publication_id` est réutilisé comme FK dans les tables dépendantes sans être recalculé.

### Ordre de chargement imposé par les FK

1. charger ou mettre à jour les sources distinctes ;
2. charger les lots distincts en référençant leur source ;
3. charger les publications en référençant leur lot ;
4. charger les images en référençant leur publication ;
5. charger uniquement les labels non nuls en référençant leur publication.

## Dictionnaire des champs finalisés

| Champ | Type final | Obligatoire | Rôle métier et usage IA |
|---|---|---:|---|
| `publication_id` | chaîne | Oui | Identifiant stable, dédoublonnage et liaison de la publication à son image. |
| `source_name` | chaîne | Oui | Source d’acquisition : NewsData.io, PolitiFact ou Fakeddit. Utile pour analyser les biais par source. |
| `source_domain` | chaîne | Oui | Domaine éditorial ou domaine cible, sans `www.`. Métadonnée de provenance. |
| `source_url` | URL HTTP(S) | Oui | URL de l’article ou du post. Traçabilité et contrôle manuel. |
| `title` | chaîne | Conditionnel | Texte court utilisable par un traitement NLP. `title` ou `text` doit être renseigné. |
| `text` | texte long | Conditionnel | Entrée NLP principale : contenu, résumé, affirmation ou titre nettoyé. `title` ou `text` doit être renseigné. |
| `image_url` | URL HTTP(S) | Oui | URL distante d’origine de l’image associée. |
| `image_path` | chemin | Oui | Chemin local de l’image validée. Garantit que le texte et l’image restent exploitables ensemble. |
| `image_size_bytes` | entier | Oui | Taille réelle du fichier. Contrôle des images vides ou modifiées. |
| `image_sha256` | chaîne hexadécimale | Oui | Empreinte du contenu de l’image, utilisée pour l’intégrité et la détection de doublons. |
| `image_provenance_status` | chaîne | Oui | Indique si le lien URL-image est `downloaded`, `metadata_verified`, `legacy_adopted` ou `legacy_unverified`. |
| `published_at` | date-heure UTC | Oui | Date de publication normalisée. Analyse temporelle et séparation chronologique de datasets. |
| `language` | chaîne ISO 639-1 | Oui | Langue sous forme `fr` ou `en`. Routage vers le bon traitement NLP. |
| `author` | chaîne | Non | Auteur ou liste d’auteurs réunie dans une chaîne. Métadonnée éditoriale. |
| `source_label_raw` | chaîne | Non | Label original sans conversion silencieuse. Peut servir de cible après interprétation documentée. |
| `source_label_scheme` | chaîne | Non | Décrit le système qui donne son sens au label brut. |
| `label_provenance` | chaîne | Non | Organisation ou méthode ayant produit le label. Indique son niveau de confiance. |
| `collected_at` | date-heure UTC | Oui | Date de collecte du lot, distincte de la date de publication. Traçabilité ETL. |

## Mapping par source

| Champ métier | NewsData.io | PolitiFact | Fakeddit |
|---|---|---|---|
| Identifiant | préfixe `newsdata_` + `article_id` | préfixe `politifact_` + hash stable de l’ID RSS | préfixe `fakeddit_` + ID Reddit |
| Titre | `title` | `title` | `title`, sinon `clean_title` |
| Texte | `content`, sinon `description`, sinon `ai_summary` | `content_html` nettoyé, sinon `summary` | `clean_title`, sinon `title` |
| Date | `pubDate` | `published` RSS | `created_utc` Unix |
| Langue | `french` devient `fr` | `en` | `en` |
| Auteur | liste `creator` réunie | `author` | `author` |
| Label | aucun label inventé | verdict du `flat-meter` PolitiFact | labels 2, 3 et 6 classes conservés dans une chaîne JSON |

## Contraintes d’intégrité

Une publication transformée est acceptée seulement si :

1. son identifiant est présent et unique ;
2. son titre ou son texte est présent ;
3. ses URLs de publication et d’image sont des URLs HTTP(S) valides ;
4. sa langue est un code ISO à deux lettres ;
5. ses dates sont convertibles en UTC ;
6. son image existe dans `data/images/`, n’est pas vide et possède une signature JPEG, PNG, GIF ou WebP reconnue ;
7. le nom du fichier image correspond à l’identifiant attendu pour la publication ;
8. la taille locale correspond à la taille enregistrée pendant Extract ;
9. pour les nouveaux lots, l’URL et le hash correspondent à la preuve créée pendant le téléchargement ;
10. les trois champs de label sont soit tous renseignés, soit tous absents ;
11. le même `publication_id` ou le même couple `source_url`–`image_sha256` n’est conservé qu’une fois.

Les publications refusées et les occurrences écartées comme doublons sont écrites dans `data/rejected/invalid_records.jsonl` avec leur source, leur identifiant brut et le motif de la décision.

Les anciens JSON créés avant l’ajout de la preuve image sont signalés par `legacy_unverified`. Ils peuvent être acceptés pour compatibilité ou refusés avec `--legacy-image-policy reject`.

## Rôle des groupes de données pour l’IA

- **NLP** : `title`, `text`, `language`.
- **Analyse d’image** : `image_path`, `image_sha256`, `image_size_bytes`.
- **Fusion multimodale** : `publication_id` relie sans ambiguïté le texte et l’image.
- **Qualité de l’association** : `image_provenance_status` indique le niveau de confiance du lien URL-image.
- **Classification supervisée éventuelle** : les trois champs `source_label_*`, après étude du sens de chaque schéma de labels.
- **Contrôle des biais et traçabilité** : source, domaine, auteur, dates et URLs.

Le projet actuel prépare les données ; il n’entraîne pas encore de modèle et ne transforme pas automatiquement tous les labels en classes `true` ou `fake`.
